# Exploration of the training data

This notebook illustrates how to load and interpret the simulations and preprocessed data.

(Create those files by running the `workflow.ipynb` notebook)

**Suggestion**: Copy this file into the desired `rundir`, then you will have the plots for each run in its relevant notebook.

In [ ]:
from pathlib import Path
import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from cogwheel import gw_plotting

from labrador import transform, utils, waveform_model

In [ ]:
rundir = Path('.')
datadir = rundir/utils.TEST_DIR

In [ ]:
config = utils.load_data_config(rundir)

detector_names = tuple(config.EVENT_DATA_KWARGS['detector_names'])
ref_det_name = config.PRIOR_KWARGS['ref_det_name']
i_refdet = detector_names.index(ref_det_name)

In [ ]:
slice_ = slice(10**4)  # To keep amount of data manageable
preprocessed_data = utils.get_preprocessed_data(datadir, slice_=slice_)

for key, val in preprocessed_data.items():
    print(key, val.shape, val.dtype)

In [ ]:
mask = np.load(datadir/'mask.npy')[slice_]

In [ ]:
simulation_parameters = pd.read_feather(datadir/'simulation_parameters.feather')[slice_][mask]

In [ ]:
summary = utils.get_summary(datadir, apply_mask=False)[slice_][mask]
processed_coef_keys = waveform_model.PhenomenologicalWaveformGenerator.\
    from_rundir(rundir).processed_coef_keys

processed_coef = pd.DataFrame(preprocessed_data['processed_coef'],
                              columns=processed_coef_keys,
                              index=summary.index)
full_df = pd.concat([summary, processed_coef], axis=1)

full_df['mtot'] = full_df['m1'] + full_df['m2']
full_df['q'] = np.exp(full_df['lnq'])
full_df = pd.concat([summary, processed_coef], axis=1)

full_df['mtot'] = full_df['m1'] + full_df['m2']
full_df['q'] = np.exp(full_df['lnq'])

In [ ]:
# h_h of injection vs reference waveform
plt.figure()
plt.plot(preprocessed_data['h_h'], preprocessed_data['h0_h0'], '.', label=detector_names)
plt.xlabel(r'Injection $\langle h \mid h \rangle$')
plt.ylabel(r'Reference waveform $\langle h_0 \mid h_0 \rangle$')
plt.loglog()
plt.grid()
plt.legend();

In [ ]:
# We might be including examples with too extreme SNR (high & low)
# The maximization would seem to be working
full_df.plot('snr', 'snr0', 'scatter', c='lnq')
plt.loglog()
plt.grid();

In [ ]:
# Calibrate the (PSD dependent) relationship between d_hat and h_h
# d_hat is supposed to be proportional to 1/snr at the reference detector assuming a pN formula for the signal amplitude.
# So, some of the scatter could be the snr in other detectors.
# There seems to be a trend with mass, probably due to the merger falling in band for high mass.
full_df.plot('amp_refdet', 'h_h', 'scatter', c='lnmchirp')
plt.loglog()
plt.grid();

In [ ]:
# Indeed, with h_h_L the relation is somewhat tighter
plt.figure()
plt.scatter(full_df['amp_refdet'],
            preprocessed_data['h_h'][:, i_refdet],
            c=full_df['lnmchirp'])
plt.loglog()
plt.grid()

plt.xlabel(r'$\hat d^{-1}$')
plt.ylabel(rf'$\langle h \mid h \rangle_{{\rm {ref_det_name}}}$');

In [ ]:
# Indeed, with h_h_L the relation is somewhat tighter
plt.figure()
plt.scatter(full_df['amp_refdet'],
            preprocessed_data['h_h'][:, i_refdet],
            c=full_df['chieff'])
plt.loglog()
plt.grid()

plt.xlabel(r'$\hat d^{-1}$')
plt.ylabel(rf'$\langle h \mid h \rangle_{{\rm {ref_det_name}}}$');

In [ ]:
# In real life there are multiple detectors and noise, so the snr gets larger fluctuations.
# It seems that this is about as good as we can get to simulating snr > 8 events by using a d_hat cut.
# Perhaps we can manually reject simulations outside some SNR range before training.
full_df.plot('amp_refdet', 'snr', 'scatter', c='lnmchirp')
plt.loglog()
plt.grid()
plt.axhline(8, ls='--', c='k')

In [ ]:
# fcut ~ 1/Mtot (once it enters the detector band).
plt.figure()
plt.scatter(full_df['mtot'],
            full_df['log10_fcut'],
            c=full_df['m2'] / full_df['m1'])
plt.colorbar(label='$q$')
plt.loglog()
plt.xlabel(r'$M_{\rm tot} (\rm M_\odot)$')
plt.ylabel(r'$f_{\rm cut}$ (Hz)')
plt.grid();

## Correlations between truth and features

In [ ]:
full_df.plot('mtot', 'log10_fcut', 'scatter', c='chieff')
plt.grid()

In [ ]:
full_df.plot('mtot', 'chieff', 'scatter', c='log10_fcut')
plt.xscale('log')

In [ ]:
folded_transform_params = config.TRANSFORM_CLASS.sampled_params.copy()
for folded_par in config.TRANSFORM_CLASS.folded_params:
    folded_transform_params[folded_transform_params.index(folded_par)] = f'folded_{folded_par}'

plot_params = folded_transform_params + processed_coef_keys

In [ ]:
gw_plotting.CornerPlot(full_df, params=plot_params).plot()

The first block are the truths we want to get posteriors for.

The second block are the processed coefficients.

The best case is if there is either no correlation (the analytic coordinate transform was so good that it contains all the information, so the coefficients are useless) or perfect correlation (the coefficients are great predictors of the parameters).

For example, there is good correlation between `cos_thetanet` and `time_difference`, as expected.

Note that there are correlations between some of the coefficients by their definition, e.g. `amp_ratio_0` and `amp_ratio_1`, or `cos_phase_difference` and `sin_phase_difference`.

### Look at relation between intrinsic parameters and reference waveform coefficients

In [ ]:
full_df.plot('intphasecoef_0', 'intphasecoef_1', 'scatter', s=5, c='lnmchirp');

In [ ]:
full_df.plot('intphasecoef_0', 'intphasecoef_1', 'scatter', s=5, c='lnq');

In [ ]:
full_df.plot('lnmchirp', 'lnq', 'scatter', c='intphasecoef_0');

In [ ]:
full_df.plot('lnmchirp', 'lnq', 'scatter', c='intphasecoef_1');

In [ ]:
full_df.plot('lnmchirp', 'chieff', 'scatter', c='intphasecoef_1');

In [ ]:
# Orbital phase
# Faint-and-massive events get it wrong?
full_df.plot('folded_phi_ref_hat', 'intphasecoef_0', 'scatter', c='snr0')
plt.grid()

In [ ]:
full_df.plot('relative_dhat', 'snr0', 'scatter', c='lnmchirp')

In [ ]:
full_df.plot('relative_dhat', 'lnmchirp', 'scatter', c='snr0')

In [ ]:
# The time difference between H and L has information about sky localization polar angle
full_df.plot('folded_phinet_hat', 'costhetanet', 'scatter', c='time_difference_1-0')

In [ ]:
# The phase difference between H and L has information about sky localization azimuth
full_df.plot('folded_phinet_hat', 'costhetanet', 'scatter', c='sin_phase_difference_1-0')

### Heterodyned data

In [ ]:
preprocessed_data['heterodyned_data'].shape  # (n_simulations, n_detectors, n_frequencies)

In [ ]:
color_key = 'chieff'
colors = full_df[color_key]
norm = mpl.colors.Normalize(colors.min(), colors.max())
cmap = plt.cm.viridis
sm = plt.cm.ScalarMappable(cmap=plt.cm.viridis, norm=norm)

fig, axs = plt.subplots(2, sharex=True, sharey=True)
for color, heterodyned_data in zip(colors, preprocessed_data['heterodyned_data']):
    axs[0].plot(#preprocessed_data['fbin'],
                heterodyned_data[i_refdet].real, lw=.5,
                color=sm.to_rgba(color), zorder=color)

    axs[1].plot(#preprocessed_data['fbin'],
                heterodyned_data[i_refdet].imag, lw=.5,
                color=sm.to_rgba(color), zorder=color)

plt.colorbar(sm, ax=axs, label=color_key)
axs[0].set_title(f'Heterodyned data at {ref_det_name}')
axs[0].set_ylabel(r'Re')
axs[1].set_ylabel(r'Im')
axs[1].set_xlabel('Coarse frequency index $b$')
axs[1].xaxis.set_major_locator(mpl.ticker.MaxNLocator(integer=True))

### Heterodyned signal (noiseless)

In [ ]:
fig, axs = plt.subplots(2, sharex=True, sharey=True)
for color, heterodyned_data in zip(colors, preprocessed_data['heterodyned_signal']):
    axs[0].plot(#preprocessed_data['fbin'],
                heterodyned_data[i_refdet].real, lw=.5,
                color=sm.to_rgba(color), zorder=color)

    axs[1].plot(#preprocessed_data['fbin'],
                heterodyned_data[i_refdet].imag, lw=.5,
                color=sm.to_rgba(color), zorder=color)

plt.colorbar(sm, ax=axs, label=color_key)
axs[0].set_title(f'Heterodyned signal at {ref_det_name}')
axs[0].set_ylabel(r'Re')
axs[1].set_ylabel(r'Im')
axs[1].set_xlabel('Coarse frequency index $b$')
axs[1].xaxis.set_major_locator(mpl.ticker.MaxNLocator(integer=True))

In [ ]:
for color_key in 'snr', 'lnmchirp', 'q', 'chieff':
    colors = full_df[color_key]
    norm = mpl.colors.Normalize(colors.min(), colors.max())
    cmap = plt.cm.viridis
    sm = plt.cm.ScalarMappable(cmap=plt.cm.viridis, norm=norm)

    plt.figure()
    for color, heterodyned_data in zip(colors, preprocessed_data['heterodyned_signal']):
        amp = np.abs(heterodyned_data[i_refdet])
        i0 = np.argmax(amp)
        phase = np.unwrap(np.angle(heterodyned_data[i_refdet]), axis=0)
        plt.scatter(
            np.arange(len(phase)),
            phase - np.round(phase[i0] / (2 * np.pi)) * 2 * np.pi,
            color=sm.to_rgba(color),
            zorder=color,
            alpha=amp / amp[i0],
            linewidths=0,
            s=5
        )


    plt.colorbar(sm, ax=plt.gca(), label=color_key)
    plt.ylim(-np.pi, np.pi)
    plt.grid()
    plt.title(f'Heterodyned signal at {ref_det_name}')
    plt.ylabel(r'Phase')
    plt.xlabel('Coarse frequency index $b$')
    plt.gca().xaxis.set_major_locator(mpl.ticker.MaxNLocator(integer=True))